# Robustness Checks Notebook

This notebook validates whether conclusions remain stable under alternative specifications and samples.

Checks:
- Ramsey RESET on forecast calibration regressions.
- Validation-vs-test error stability tests.
- Tail-event exclusion sensitivity for DM tests.

In [13]:
import os
import sys
import numpy as np
import pandas as pd
from scipy import stats

if os.path.basename(os.getcwd()) == 'khanh_model_analysis':
    os.chdir('..')

sys.path.append('src')

from statistical_validation import (
    set_reproducible_seed,
    load_config,
    discover_forecasts,
    reset_specification_test,
    run_dm_comparisons,
    build_default_dm_pairs,
)

set_reproducible_seed(42)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 200)

#### Interpretation
This setup cell prepares robustness utilities. Its role is to ensure all follow-up tests are run under the same package versions and helper definitions.

In [14]:
config = load_config('configs/pipeline_config.yaml')
active_target = config['active_target']
forecasts = discover_forecasts(config)

print(f'Active target: {active_target}')
print('Rows:', len(forecasts))
print('Models:', sorted(forecasts['Model'].unique().tolist()))

Active target: PHP
Rows: 63450
Models: ['arima', 'arimax', 'baseline_ar1', 'baseline_mean', 'baseline_rw', 'hybrid_arima_mlp', 'hybrid_arima_svr', 'hybrid_arimax_mlp', 'hybrid_arimax_svr', 'hybrid_var_mlp', 'hybrid_var_svr', 'hybrid_varx_mlp', 'hybrid_varx_svr', 'var', 'varx']


#### Interpretation
The printed metadata verifies active target, sample size, and model count. These checks protect robustness conclusions from accidental data-path mismatch.

### Why robustness matters

A model can look strong on one split and still fail under small specification changes. The checks below ask whether conclusions survive a RESET test, whether validation errors behave similarly on the test set, and whether DM conclusions are driven by a few extreme tail observations. That is the right standard for a forecasting pipeline that may later expand to ARIMAX, VARX, or new hybrid variants.

## 1) Ramsey RESET (Specification Bias)

Null: no omitted non-linear terms in calibration equation `Actual ~ Forecast`.

In [15]:
reset_val = reset_specification_test(forecasts, set_name='val')
reset_test = reset_specification_test(forecasts, set_name='test')

display(reset_val.sort_values(['Pair', 'RESET_pvalue']))
display(reset_test.sort_values(['Pair', 'RESET_pvalue']))

reset_compare = reset_val.merge(
    reset_test,
    on=['Pair', 'Model'],
    suffixes=('_val', '_test'),
)
reset_compare['stable_conclusion'] = (
    reset_compare['Specification_bias_5pct_val'] == reset_compare['Specification_bias_5pct_test']
)
display(reset_compare[['Pair', 'Model', 'RESET_pvalue_val', 'RESET_pvalue_test', 'stable_conclusion']])

,Pair,Model,Set,R2,RESET_F,RESET_pvalue,Specification_bias_5pct,RESET_note
5,CNYPHP_RET,hybrid_arima_mlp,val,0.088081,22.879028,0.000002,True,OK
6,CNYPHP_RET,hybrid_arima_svr,val,0.017301,21.269198,0.000005,True,OK
7,CNYPHP_RET,hybrid_arimax_mlp,val,0.104905,19.214939,0.000015,True,OK
8,CNYPHP_RET,hybrid_arimax_svr,val,0.040494,19.060807,0.000016,True,OK
0,CNYPHP_RET,arima,val,0.099395,9.426465,0.002278,True,OK
1,CNYPHP_RET,arimax,val,0.112306,9.256289,0.002494,True,OK
2,CNYPHP_RET,baseline_ar1,val,0.082906,7.960822,0.005007,True,OK
14,CNYPHP_RET,varx,val,0.011156,4.413658,0.036248,True,OK
13,CNYPHP_RET,var,val,0.003700,2.973281,0.085386,False,OK
10,CNYPHP_RET,hybrid_var_svr,val,0.001677,1.693737,0.193822,False,OK


,Pair,Model,Set,R2,RESET_F,RESET_pvalue,Specification_bias_5pct,RESET_note
14,CNYPHP_RET,varx,test,0.169698,59.904960,7.494021e-14,True,OK
13,CNYPHP_RET,var,test,0.128851,55.403649,5.604828e-13,True,OK
1,CNYPHP_RET,arimax,test,0.198448,46.016788,3.998417e-11,True,OK
2,CNYPHP_RET,baseline_ar1,test,0.114140,43.764347,1.130262e-10,True,OK
0,CNYPHP_RET,arima,test,0.148077,39.980309,6.567532e-10,True,OK
9,CNYPHP_RET,hybrid_var_mlp,test,0.207689,21.214998,5.450566e-06,True,OK
11,CNYPHP_RET,hybrid_varx_mlp,test,0.252342,18.149940,2.522287e-05,True,OK
10,CNYPHP_RET,hybrid_var_svr,test,0.186407,10.166515,1.537193e-03,True,OK
5,CNYPHP_RET,hybrid_arima_mlp,test,0.237579,4.349233,3.762875e-02,True,OK
7,CNYPHP_RET,hybrid_arimax_mlp,test,0.287697,3.573162,5.940860e-02,False,OK


,Pair,Model,RESET_pvalue_val,RESET_pvalue_test,stable_conclusion
0,CNYPHP_RET,arima,0.002278,6.567532e-10,True
1,CNYPHP_RET,arimax,0.002494,3.998417e-11,True
2,CNYPHP_RET,baseline_ar1,0.005007,1.130262e-10,True
3,CNYPHP_RET,baseline_mean,NaN,NaN,True
4,CNYPHP_RET,baseline_rw,NaN,NaN,True
5,CNYPHP_RET,hybrid_arima_mlp,0.000002,3.762875e-02,True
6,CNYPHP_RET,hybrid_arima_svr,0.000005,7.139883e-01,False
7,CNYPHP_RET,hybrid_arimax_mlp,0.000015,5.940860e-02,False
8,CNYPHP_RET,hybrid_arimax_svr,0.000016,5.297155e-01,False
9,CNYPHP_RET,hybrid_var_mlp,0.663689,5.450566e-06,False


#### Interpretation
RESET results test functional-form adequacy. Rejection implies potential misspecification and motivates flexible nonlinear components; non-rejection supports the baseline functional form.

## 2) Validation vs Test Error Stability

A robust model should not deteriorate sharply when moving from validation to test.

In [16]:
stability_rows = []
for (pair, model), chunk in forecasts.groupby(['Pair', 'Model']):
    val_abs = chunk.loc[chunk['Set'] == 'val', 'AE'].dropna().values
    test_abs = chunk.loc[chunk['Set'] == 'test', 'AE'].dropna().values
    if len(val_abs) < 10 or len(test_abs) < 10:
        continue

    mw = stats.mannwhitneyu(val_abs, test_abs, alternative='two-sided')
    stability_rows.append({
        'Pair': pair,
        'Model': model,
        'Val_MAE': float(np.mean(val_abs)),
        'Test_MAE': float(np.mean(test_abs)),
        'Delta_Test_minus_Val': float(np.mean(test_abs) - np.mean(val_abs)),
        'MW_pvalue': float(mw.pvalue),
        'Distribution_shift_5pct': float(mw.pvalue) < 0.05,
    })

stability_df = pd.DataFrame(stability_rows).sort_values(['Pair', 'Delta_Test_minus_Val'])
display(stability_df)

,Pair,Model,Val_MAE,Test_MAE,Delta_Test_minus_Val,MW_pvalue,Distribution_shift_5pct
12,CNYPHP_RET,hybrid_varx_svr,0.418450,0.375845,-0.042606,0.251288,False
11,CNYPHP_RET,hybrid_varx_mlp,0.406809,0.374367,-0.032442,0.316465,False
10,CNYPHP_RET,hybrid_var_svr,0.425791,0.393824,-0.031967,0.476342,False
8,CNYPHP_RET,hybrid_arimax_svr,0.401076,0.369996,-0.031080,0.236601,False
7,CNYPHP_RET,hybrid_arimax_mlp,0.392226,0.365362,-0.026865,0.181261,False
6,CNYPHP_RET,hybrid_arima_svr,0.407449,0.382081,-0.025368,0.312135,False
9,CNYPHP_RET,hybrid_var_mlp,0.409594,0.387458,-0.022135,0.573202,False
14,CNYPHP_RET,varx,0.402555,0.383369,-0.019186,0.505379,False
5,CNYPHP_RET,hybrid_arima_mlp,0.396912,0.377803,-0.019109,0.291328,False
13,CNYPHP_RET,var,0.404971,0.394842,-0.010129,0.790956,False


#### Interpretation
Validation-versus-test stability compares model behavior across splits. Small drift in metrics indicates robust generalization; large drift suggests overfitting or regime sensitivity.

## 3) Tail-Event Exclusion Sensitivity (DM)

We remove the top 1% absolute forecast errors by pair/model and re-run DM tests to verify conclusions are not driven only by a few extreme events.

In [17]:
trimmed = forecasts.copy()
trimmed['keep'] = True

for (pair, model), chunk in trimmed.groupby(['Pair', 'Model']):
    q = chunk['AE'].quantile(0.99)
    idx = chunk[chunk['AE'] > q].index
    trimmed.loc[idx, 'keep'] = False

trimmed = trimmed[trimmed['keep']].drop(columns=['keep'])

models = sorted(trimmed['Model'].unique().tolist())
dm_pairs = build_default_dm_pairs(models)

full_dm = run_dm_comparisons(forecasts, dm_pairs, criterion='mse', set_name='test')
trim_dm = run_dm_comparisons(trimmed, dm_pairs, criterion='mse', set_name='test')

merged_dm = full_dm.merge(
    trim_dm,
    on=['Pair', 'Model_A', 'Model_B', 'Loss', 'Set'],
    suffixes=('_full', '_trim')
)
merged_dm['same_significance'] = (
    (merged_dm['p_value_full'] < 0.05) == (merged_dm['p_value_trim'] < 0.05)
)
display(merged_dm.sort_values(['Pair', 'p_value_trim']))

,Pair,Model_A,Model_B,Loss,Set,DM_stat_full,p_value_full,n_obs_full,A_better_than_B_5pct_full,DM_stat_trim,p_value_trim,n_obs_trim,A_better_than_B_5pct_trim,same_significance
7,CNYPHP_RET,arimax,baseline_ar1,mse,test,-4.360069,1.635611e-05,423,True,-3.688673,2.554173e-04,416,True,True
8,CNYPHP_RET,arimax,baseline_mean,mse,test,-3.213419,1.412277e-03,423,True,-3.474125,5.663300e-04,417,True,True
9,CNYPHP_RET,arimax,baseline_rw,mse,test,-3.199968,1.478209e-03,423,True,-3.469661,5.755545e-04,417,True,True
16,CNYPHP_RET,hybrid_arimax_mlp,baseline_ar1,mse,test,-2.418999,1.598536e-02,423,True,-3.238139,1.299791e-03,416,True,True
17,CNYPHP_RET,hybrid_arimax_mlp,baseline_mean,mse,test,-2.457297,1.440006e-02,423,True,-2.908358,3.828463e-03,416,True,True
18,CNYPHP_RET,hybrid_arimax_mlp,baseline_rw,mse,test,-2.453043,1.456901e-02,423,True,-2.906121,3.855389e-03,416,True,True
38,CNYPHP_RET,varx,baseline_mean,mse,test,-2.711602,6.969376e-03,423,True,-2.403519,1.667486e-02,417,True,True
39,CNYPHP_RET,varx,baseline_rw,mse,test,-2.701946,7.171426e-03,423,True,-2.402327,1.672856e-02,417,True,True
4,CNYPHP_RET,arima,baseline_ar1,mse,test,-2.916155,3.732740e-03,423,True,-2.320902,2.077594e-02,416,True,True
6,CNYPHP_RET,arima,baseline_rw,mse,test,-2.587261,1.000769e-02,423,True,-2.293753,2.230364e-02,416,True,True


#### Interpretation
Tail-sensitivity DM analysis checks whether conclusions hold after trimming extremes. If significance patterns persist, relative model ranking is not driven only by outlier episodes.

In [18]:
out_dir = f'results/{active_target}/evaluation'
os.makedirs(out_dir, exist_ok=True)

reset_test.to_csv(f'{out_dir}/reset_test_results.csv', index=False)
stability_df.to_csv(f'{out_dir}/val_test_error_stability.csv', index=False)
merged_dm.to_csv(f'{out_dir}/dm_tail_sensitivity.csv', index=False)

print('Saved robustness outputs to', out_dir)

Saved robustness outputs to results/PHP/evaluation


#### Interpretation
This save step records robustness tables for reproducible reporting. Stored outputs should match the tables displayed above to ensure consistency.